In [2]:
import glob
import json
import os
import re
import xml.etree.ElementTree as ET
import fiona
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import LineString, MultiLineString, MultiPolygon, Polygon
from shapely.ops import unary_union

# Enable KML driver support in Fiona
fiona.drvsupport.supported_drivers["KML"] = "RAW"


def extract_operational_date(row):
    """Priority search for actual operational/activity date over server upload date."""
    field_date_cols = [
        "date_ploughed",
        "date",
        "today",
        "start",
        "end",
        "Date",
        "Date_Ploughed",
    ]
    for col in field_date_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
            parsed = pd.to_datetime(row[col], errors="coerce")
            if pd.notna(parsed):
                return parsed.strftime("%Y-%m-%d")

    text_to_search = f"{row.get('Comments', '')} {row.get('notes', '')}"
    found_dates = re.findall(
        r"\b(20\d{2}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12][0-9]|3[01]))\b",
        text_to_search,
    )
    if found_dates:
        return found_dates[0]

    meta_cols = ["Submission_Date", "_submission_time"]
    for col in meta_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip() != "":
            parsed = pd.to_datetime(row[col], errors="coerce")
            if pd.notna(parsed):
                return parsed.strftime("%Y-%m-%d")

    return "Unknown"


def normalize_name(name):
    if pd.isna(name) or not str(name).strip():
        return ""
    s = str(name).strip().title()
    typos = {
        r"\bKelivin\b": "Kelvin",
        r"\bKevin\b": "Kelvin",
    }
    for typo, correction in typos.items():
        s = re.sub(typo, correction, s, flags=re.IGNORECASE)
    return s


def mode_or_first(series):
    valid = series[
        (series.notna())
        & (series != "")
        & (~series.astype(str).str.lower().isin(["nan", "none", "unknown"]))
    ]
    if not valid.empty:
        return valid.mode().iloc[0]
    return ""


def fix_invalid_geometries(gdf):
    """Fixes invalid geometry topology using shapely.make_valid and buffer(0)."""
    if gdf is None or gdf.empty:
        return gdf
    gdf = gdf.copy()
    gdf["geometry"] = gdf["geometry"].apply(
        lambda g: (
            shapely.make_valid(g)
            if (g is not None and not g.is_valid)
            else g
        )
    )
    gdf["geometry"] = gdf["geometry"].apply(
        lambda g: (
            g.buffer(0) if (g is not None and not g.is_valid) else g
        )
    )
    return gdf


def clean_and_standardize_data(gdf):
    text_cols = [
        "County",
        "Subcounty",
        "Ward",
        "Location",
        "Sublocation",
        "Village",
        "Farmer Name",
        "Field Officer",
        "Tractor Owner",
        "Tractor Driver",
        "Tractor Driver - Number Plate",
        "Seed Variety",
        "Soil Type",
        "Farm Type",
        "Comments",
    ]

    for col in text_cols:
        if col in gdf.columns:
            gdf[col] = gdf[col].apply(normalize_name)
        else:
            gdf[col] = ""

    def resolve_county_group(row):
        c = row["County"]
        sc = row["Subcounty"]
        w = row["Ward"]
        if "Kibwezi East" in [c, sc, w] or sc == "Kibwezi East":
            return "Kibwezi East"
        return c if c != "" else "Makueni"

    gdf["County_Group"] = gdf.apply(resolve_county_group, axis=1)
    gdf["Clean_Date"] = gdf.apply(extract_operational_date, axis=1)

    driver_plate_map = (
        gdf[gdf["Tractor Driver"] != ""]
        .groupby("Tractor Driver")["Tractor Driver - Number Plate"]
        .apply(mode_or_first)
        .to_dict()
    )
    driver_owner_map = (
        gdf[gdf["Tractor Driver"] != ""]
        .groupby("Tractor Driver")["Tractor Owner"]
        .apply(mode_or_first)
        .to_dict()
    )

    gdf["Tractor Driver - Number Plate"] = gdf.apply(
        lambda r: (
            driver_plate_map.get(
                r["Tractor Driver"], r["Tractor Driver - Number Plate"]
            )
            if r["Tractor Driver - Number Plate"] == ""
            else r["Tractor Driver - Number Plate"]
        ),
        axis=1,
    )

    gdf["Tractor Owner"] = gdf.apply(
        lambda r: (
            driver_owner_map.get(r["Tractor Driver"], r["Tractor Owner"])
            if r["Tractor Owner"] == ""
            else r["Tractor Owner"]
        ),
        axis=1,
    )

    officer_location_map = (
        gdf[gdf["Field Officer"] != ""]
        .groupby(["Ward", "Tractor Driver"])["Field Officer"]
        .apply(mode_or_first)
        .to_dict()
    )

    def resolve_officer_name(row):
        officer = row["Field Officer"]
        if len(officer.split()) == 1 and officer != "":
            key = (row["Ward"], row["Tractor Driver"])
            full_name = officer_location_map.get(key, officer)
            if full_name and officer.lower() in full_name.lower():
                return full_name
        return officer

    gdf["Field Officer"] = gdf.apply(resolve_officer_name, axis=1)

    for col in gdf.columns:
        if pd.api.types.is_datetime64_any_dtype(gdf[col]):
            gdf[col] = (
                pd.to_datetime(gdf[col], errors="coerce")
                .dt.strftime("%Y-%m-%d %H:%M:%S")
                .fillna("")
            )

    gdf = fix_invalid_geometries(gdf)
    gdf_proj = gdf.to_crs(epsg=8857)
    gdf["Calculated_Acres"] = (gdf_proj.geometry.area / 4046.8564224).round(2)

    if "Acres" in gdf.columns:
        gdf["Acres"] = (
            pd.to_numeric(gdf["Acres"], errors="coerce")
            .fillna(gdf["Calculated_Acres"])
            .round(2)
        )
    else:
        gdf["Acres"] = gdf["Calculated_Acres"]

    gdf["_record_id"] = gdf.index.astype(str)

    tractor_stats = (
        gdf.groupby(
            [
                "Field Officer",
                "Clean_Date",
                "Tractor Driver",
                "Tractor Driver - Number Plate",
            ],
            observed=False,
        )
        .agg(
            Tractor_Acres=("Acres", "sum"),
            Tractor_Farms=("_record_id", "nunique"),
        )
        .reset_index()
    )

    officer_daily_map = {}
    for (officer, date_val), group in tractor_stats.groupby(
        ["Field Officer", "Clean_Date"], observed=False
    ):
        sub_slice = gdf[
            (gdf["Field Officer"] == officer) & (gdf["Clean_Date"] == date_val)
        ]
        total_acres = sub_slice["Acres"].sum()

        total_farmers = (
            sub_slice["Farmer Name"]
            .replace("", pd.NA)
            .dropna()
            .nunique()
        )
        if total_farmers == 0:
            total_farmers = len(sub_slice)

        breakdown_list = []
        for _, r in group.iterrows():
            drv = r["Tractor Driver"] or "Unknown Driver"
            plt = r["Tractor Driver - Number Plate"]
            plt_str = f" ({plt})" if plt else ""
            breakdown_list.append(
                f"• {drv}{plt_str}: <strong>{r['Tractor_Acres']:.2f} Acres</strong> ({r['Tractor_Farms']} farms)"
            )

        officer_daily_map[(officer, date_val)] = {
            "Total_Acres": f"{total_acres:.2f}",
            "Unique_Farmers": total_farmers,
            "Tractor_Breakdown": "<br>".join(breakdown_list),
        }

    gdf["Total_Acres"] = gdf.apply(
        lambda r: officer_daily_map.get(
            (r["Field Officer"], r["Clean_Date"]), {}
        ).get("Total_Acres", "0.00"),
        axis=1,
    )
    gdf["Unique_Farmers"] = gdf.apply(
        lambda r: officer_daily_map.get(
            (r["Field Officer"], r["Clean_Date"]), {}
        ).get("Unique_Farmers", 0),
        axis=1,
    )
    gdf["Tractor_Breakdown"] = gdf.apply(
        lambda r: officer_daily_map.get(
            (r["Field Officer"], r["Clean_Date"]), {}
        ).get("Tractor_Breakdown", ""),
        axis=1,
    )

    return gdf


def get_distinct_color(index, total_count):
    hue = (index * (360 / max(total_count, 1))) % 360
    return f"hsl({hue:.1f}, 80%, 45%)"


def parse_kml_xml_safely(file_path):
    """Directly parses KML XML structure to rescue track geometries."""
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
    except Exception as e:
        print(f"Failed to XML-parse file '{file_path}': {e}")
        return []

    ns = {"kml": "http://www.opengis.net/kml/2.2"}
    filename = os.path.splitext(os.path.basename(file_path))[0]
    records = []

    placemarks = (
        root.findall(".//kml:Placemark", ns)
        if root.tag.endswith("kml")
        else root.findall(".//Placemark")
    )

    for pm in placemarks:
        name_node = (
            pm.find("kml:name", ns)
            if root.tag.endswith("kml")
            else pm.find("name")
        )
        track_id = (
            name_node.text.strip()
            if (name_node is not None and name_node.text)
            else filename
        )

        coord_nodes = (
            pm.findall(".//kml:coordinates", ns)
            if root.tag.endswith("kml")
            else pm.findall(".//coordinates")
        )

        lines = []
        for cnode in coord_nodes:
            if not cnode.text:
                continue

            raw_coords = cnode.text.strip().split()
            pts = []
            for pt in raw_coords:
                parts = pt.split(",")
                if len(parts) >= 2:
                    try:
                        lon, lat = float(parts[0]), float(parts[1])
                        pts.append((lon, lat))
                    except ValueError:
                        continue

            if len(pts) > 1:
                lines.append(LineString(pts))

        if not lines:
            continue

        geom = lines[0] if len(lines) == 1 else MultiLineString(lines)
        records.append({"Tractor_ID": track_id, "geometry": geom})

    return records


def load_tractor_kmls(kml_folder_pattern="tractor_kmls*"):
    """Robust loading of KML files with direct XML fallback parsing."""
    pattern_paths = [
        os.path.join(kml_folder_pattern, "*.kml"),
        os.path.join(kml_folder_pattern, "**", "*.kml"),
    ]
    kml_files = set()
    for p in pattern_paths:
        kml_files.update(glob.glob(p, recursive=True))

    if not kml_files:
        print(f"Warning: No .kml files found matching pattern '{kml_folder_pattern}'.")
        return None, "{}"

    print(f"Processing {len(kml_files)} KML track files...")
    combined_gdfs = []

    for file_path in kml_files:
        filename = os.path.splitext(os.path.basename(file_path))[0]

        try:
            kml_gdf = gpd.read_file(file_path, driver="KML")
            if not kml_gdf.empty:
                kml_gdf = kml_gdf[
                    kml_gdf.geometry.type.isin(["LineString", "MultiLineString"])
                ]
                kml_gdf = kml_gdf[
                    kml_gdf.geometry.apply(
                        lambda g: g is not None and len(getattr(g, "coords", [0, 0])) > 1
                    )
                ]

                if not kml_gdf.empty:
                    if "Name" in kml_gdf.columns and kml_gdf["Name"].notna().any():
                        kml_gdf["Tractor_ID"] = kml_gdf["Name"].apply(
                            lambda n: str(n) if pd.notna(n) and str(n).strip() != "" else filename
                        )
                    else:
                        kml_gdf["Tractor_ID"] = filename

                    combined_gdfs.append(kml_gdf[["Tractor_ID", "geometry"]])
                    continue
        except Exception:
            pass

        parsed_records = parse_kml_xml_safely(file_path)
        if parsed_records:
            fallback_gdf = gpd.GeoDataFrame(parsed_records, crs="EPSG:4326")
            combined_gdfs.append(fallback_gdf)

    if not combined_gdfs:
        print("Error: No valid line track data could be recovered from KML files.")
        return None, "{}"

    full_kml_gdf = pd.concat(combined_gdfs, ignore_index=True)

    if full_kml_gdf.crs is None or full_kml_gdf.crs.to_epsg() != 4326:
        full_kml_gdf = full_kml_gdf.set_crs(epsg=4326, allow_override=True).to_crs(epsg=4326)

    full_kml_gdf = fix_invalid_geometries(full_kml_gdf)

    unique_tractors = list(full_kml_gdf["Tractor_ID"].unique())
    num_tractors = len(unique_tractors)
    color_map = {
        tractor: get_distinct_color(idx, num_tractors)
        for idx, tractor in enumerate(unique_tractors)
    }

    full_kml_gdf["Track_Color"] = full_kml_gdf["Tractor_ID"].map(color_map)

    print(
        f"Successfully processed {len(full_kml_gdf)} track elements across {num_tractors} unique tractors/files."
    )
    return full_kml_gdf, full_kml_gdf.to_json()


def run_geospatial_audit_reports(gdf_farms, kml_gdf):
    """Performs spatial analysis and includes Tractor Number Plates in all CSV reports."""
    farms_proj = fix_invalid_geometries(gdf_farms.to_crs(epsg=8857))

    swath_buffer_meters = 2.5
    track_proximity_buffer_meters = 5.0

    if kml_gdf is not None and not kml_gdf.empty:
        kml_proj = fix_invalid_geometries(kml_gdf.to_crs(epsg=8857))
        tracks_union = shapely.make_valid(unary_union(kml_proj.geometry))
        swaths_union = shapely.make_valid(
            unary_union(kml_proj.geometry.buffer(swath_buffer_meters))
        )
    else:
        tracks_union = None
        swaths_union = None

    coverage_list = []
    unvisited_list = []

    for idx, farm in farms_proj.iterrows():
        geom = farm.geometry
        if geom is None or geom.is_empty:
            continue
        if not geom.is_valid:
            geom = shapely.make_valid(geom)

        area_m2 = geom.area
        tot_acres = farm["Acres"]

        ploughed_m2 = 0.0
        has_track = False

        if swaths_union is not None and not swaths_union.is_empty:
            try:
                intersect_swath = geom.intersection(swaths_union)
                ploughed_m2 = intersect_swath.area
            except Exception:
                ploughed_m2 = 0.0

            try:
                has_track = geom.intersects(tracks_union) or geom.buffer(
                    track_proximity_buffer_meters
                ).intersects(tracks_union)
            except Exception:
                has_track = False

            pct_ploughed = min(
                round((ploughed_m2 / max(area_m2, 1.0)) * 100, 2), 100.0
            )
        else:
            pct_ploughed = 0.0
            has_track = False

        status = (
            "Fully Ploughed"
            if pct_ploughed >= 60
            else ("Partially Ploughed" if pct_ploughed > 0 else "No Track")
        )

        rec = {
            "Record_ID": farm.get("_record_id", str(idx)),
            "Farmer_Name": farm.get("Farmer Name", ""),
            "County_Group": farm.get("County_Group", ""),
            "Subcounty": farm.get("Subcounty", ""),
            "Ward": farm.get("Ward", ""),
            "Field_Officer": farm.get("Field Officer", ""),
            "Tractor_Driver": farm.get("Tractor Driver", ""),
            "Tractor_Driver_Plate": farm.get("Tractor Driver - Number Plate", ""),
            "Clean_Date": farm.get("Clean_Date", ""),
            "Reported_Acres": tot_acres,
            "Coverage_Percentage": pct_ploughed,
            "Plough_Status": status,
            "Track_Visible": "Yes" if has_track else "No",
        }
        coverage_list.append(rec)

        if not has_track:
            unvisited_list.append(
                {
                    "Record_ID": rec["Record_ID"],
                    "Farmer_Name": rec["Farmer_Name"],
                    "County_Group": rec["County_Group"],
                    "Subcounty": rec["Subcounty"],
                    "Ward": rec["Ward"],
                    "Field_Officer": rec["Field_Officer"],
                    "Tractor_Driver": rec["Tractor_Driver"],
                    "Tractor_Driver_Plate": rec["Tractor_Driver_Plate"],
                    "Clean_Date": rec["Clean_Date"],
                    "Reported_Acres": rec["Reported_Acres"],
                    "Audit_Issue": "Ploughed in metadata but NO tractor tracks visible",
                }
            )

    # Duplicate geometries check
    farms_proj["wkt_geom"] = farms_proj.geometry.apply(
        lambda g: g.wkt if g else ""
    )
    dup_mask = farms_proj.duplicated(
        subset=["wkt_geom"], keep=False
    ) & (farms_proj["wkt_geom"] != "")
    dup_farms = farms_proj[dup_mask]

    exact_dup_list = []
    for _, r in dup_farms.iterrows():
        exact_dup_list.append(
            {
                "Record_ID": r.get("_record_id", ""),
                "Farmer_Name": r.get("Farmer Name", ""),
                "County_Group": r.get("County_Group", ""),
                "Subcounty": r.get("Subcounty", ""),
                "Ward": r.get("Ward", ""),
                "Field_Officer": r.get("Field Officer", ""),
                "Tractor_Driver": r.get("Tractor Driver", ""),
                "Tractor_Driver_Plate": r.get("Tractor Driver - Number Plate", ""),
                "Clean_Date": r.get("Clean_Date", ""),
                "Reported_Acres": r.get("Acres", 0),
            }
        )

    # Overlapping lands check
    sindex = farms_proj.sindex
    overlap_list = []
    seen_pairs = set()

    for i, farm_a in farms_proj.iterrows():
        geom_a = farm_a.geometry
        if geom_a is None or geom_a.is_empty:
            continue
        area_a = geom_a.area
        if area_a <= 0:
            continue

        possible_matches_index = list(sindex.intersection(geom_a.bounds))
        possible_matches = farms_proj.iloc[possible_matches_index]

        for j, farm_b in possible_matches.iterrows():
            if i >= j:
                continue

            pair_key = (min(i, j), max(i, j))
            if pair_key in seen_pairs:
                continue

            geom_b = farm_b.geometry
            if geom_b is None or geom_b.is_empty:
                continue
            area_b = geom_b.area
            if area_b <= 0:
                continue

            try:
                if geom_a.intersects(geom_b):
                    inter_area = geom_a.intersection(geom_b).area
                    min_area = min(area_a, area_b)
                    overlap_ratio = inter_area / max(min_area, 1.0)

                    if overlap_ratio > 0.05:
                        seen_pairs.add(pair_key)
                        overlap_list.append(
                            {
                                "Farmer_A_ID": farm_a.get("_record_id", ""),
                                "Farmer_A_Name": farm_a.get("Farmer Name", ""),
                                "Farmer_A_Plate": farm_a.get("Tractor Driver - Number Plate", ""),
                                "Farmer_B_ID": farm_b.get("_record_id", ""),
                                "Farmer_B_Name": farm_b.get("Farmer Name", ""),
                                "Farmer_B_Plate": farm_b.get("Tractor Driver - Number Plate", ""),
                                "County_Group": farm_a.get("County_Group", ""),
                                "Clean_Date": farm_a.get("Clean_Date", ""),
                                "Overlap_Percentage": round(overlap_ratio * 100, 2),
                                "Overlapping_Acres": round(
                                    inter_area / 4046.8564224, 2
                                ),
                            }
                        )
            except Exception:
                continue

    # Unmapped ploughed tracks check
    unmapped_tracks_list = []
    if kml_gdf is not None and not kml_gdf.empty:
        all_farms_union = shapely.make_valid(unary_union(farms_proj.geometry))
        kml_proj = fix_invalid_geometries(kml_gdf.to_crs(epsg=8857))

        for idx, kline in kml_proj.iterrows():
            track_geom = kline.geometry
            if track_geom is None or track_geom.is_empty:
                continue

            try:
                if all_farms_union is not None and not all_farms_union.is_empty:
                    unmapped_part = track_geom.difference(
                        all_farms_union.buffer(10)
                    )
                else:
                    unmapped_part = track_geom

                if not unmapped_part.is_empty and unmapped_part.length > 300:
                    est_swath_area = unmapped_part.buffer(
                        swath_buffer_meters
                    ).area
                    est_acres = round(est_swath_area / 4046.8564224, 2)

                    if est_acres >= 0.2:
                        unmapped_tracks_list.append(
                            {
                                "Unmapped_Track_ID": f"TRK_UNM_{idx}",
                                "Tractor_ID": kline.get("Tractor_ID", "Unknown Tractor"),
                                "Tractor_Owner": kline.get("Tractor Owner", "N/A"),
                                "Tractor_Driver": kline.get("Tractor Driver", "N/A"),
                                "Tractor_Driver_Plate": kline.get("Tractor Driver - Number Plate", "N/A"),
                                "Field_Officer": kline.get("Field Officer", "N/A"),
                                "Estimated_Ploughed_Acres": est_acres,
                                "Track_Length_Meters": round(unmapped_part.length, 1),
                                "Clean_Date": extract_operational_date(kline),
                            }
                        )
            except Exception:
                continue

    return {
        "coverage_report": coverage_list,
        "unvisited_report": unvisited_list,
        "duplicate_report": exact_dup_list,
        "overlap_report": overlap_list,
        "unmapped_report": unmapped_tracks_list,
    }


def generate_farm_map(
    input_gpkg_path="farms_polygons_sep_week123.gpkg",
    kml_folder_pattern="tractor_kmls*",
    output_html_path="farm_dashboard_sep_week12.html",
):
    try:
        gdf = gpd.read_file(input_gpkg_path, layer="farm_boundaries")
    except Exception:
        gdf = gpd.read_file(input_gpkg_path)

    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    gdf = clean_and_standardize_data(gdf)
    gdf = gdf.fillna("")

    kml_gdf, kml_geojson_str = load_tractor_kmls(kml_folder_pattern)
    audit_reports = run_geospatial_audit_reports(gdf, kml_gdf)

    geojson_str = gdf.to_json()
    reports_json_str = json.dumps(audit_reports)

    html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Land Preparation Interactive Dashboard</title>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    
    <style>
        body {{ margin: 0; padding: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }}
        #map {{ position: absolute; top: 0; bottom: 0; width: 100%; }}
        
        .top-bar {{
            position: absolute;
            top: 15px;
            left: 60px;
            z-index: 1000;
        }}

        .report-btn {{
            background: #1f4e78;
            color: white;
            border: none;
            padding: 10px 18px;
            font-weight: bold;
            font-size: 13px;
            border-radius: 6px;
            cursor: pointer;
            box-shadow: 0 4px 10px rgba(0,0,0,0.3);
            transition: background 0.2s;
        }}
        .report-btn:hover {{ background: #153654; }}

        .filter-panel {{
            position: absolute;
            top: 15px;
            right: 15px;
            z-index: 1000;
            background: rgba(255, 255, 255, 0.95);
            padding: 15px 20px;
            border-radius: 8px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.25);
            width: 290px;
        }}
        
        .summary-panel {{
            position: absolute;
            bottom: 20px;
            left: 20px;
            z-index: 1000;
            background: rgba(255, 255, 255, 0.98);
            padding: 15px;
            border-radius: 8px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.3);
            max-width: 450px;
            max-height: 320px;
            overflow-y: auto;
        }}

        .filter-panel h4, .summary-panel h4 {{ margin: 0 0 10px 0; color: #1f4e78; font-size: 15px; border-bottom: 2px solid #2980b9; padding-bottom: 4px; }}
        .filter-group {{ margin-bottom: 10px; }}
        .filter-group label {{ display: block; font-size: 11px; font-weight: bold; margin-bottom: 3px; color: #34495e; }}
        .filter-group select, .filter-group input {{ width: 100%; padding: 6px 8px; border-radius: 4px; border: 1px solid #bdc3c7; background: #fff; font-size: 12px; box-sizing: border-box; }}
        
        .reset-btn {{
            width: 100%;
            padding: 7px;
            background-color: #e74c3c;
            color: white;
            border: none;
            border-radius: 4px;
            cursor: pointer;
            font-weight: bold;
            font-size: 12px;
            margin-top: 5px;
        }}
        .reset-btn:hover {{ background-color: #c0392b; }}

        .summary-table {{ width: 100%; border-collapse: collapse; font-size: 11px; text-align: left; }}
        .summary-table th {{ background-color: #1f4e78; color: white; padding: 6px; border: 1px solid #1f4e78; }}
        .summary-table td {{ padding: 5px; border: 1px solid #d0d7de; }}
        .summary-table tr.county-header {{ background-color: #f2f4f8; font-weight: bold; color: #1f4e78; }}
        .summary-table tr.county-subtotal {{ background-color: #eaf2f8; font-weight: bold; border-top: 2px solid #1f4e78; border-bottom: 2px solid #1f4e78; }}
        .summary-table tr.grand-total {{ background-color: #d4efdf; font-weight: bold; border-top: 2px solid #27ae60; color: #196f3d; }}

        .popup-card {{ font-size: 12px; line-height: 1.4; }}
        .popup-card h3 {{ margin: 0 0 6px 0; color: #27ae60; border-bottom: 1px solid #ddd; padding-bottom: 3px; }}
        .officer-note {{
            background-color: #ebf5fb;
            border-left: 4px solid #2980b9;
            padding: 8px;
            margin: 6px 0;
            border-radius: 3px;
        }}
        .field-label {{ font-weight: bold; color: #555; }}

        .leaflet-tractor-line {{
            pointer-events: none !important;
        }}

        .modal-overlay {{
            display: none;
            position: fixed;
            top: 0; left: 0; width: 100%; height: 100%;
            background: rgba(0,0,0,0.6);
            z-index: 2000;
            justify-content: center;
            align-items: center;
        }}

        .modal-content {{
            background: white;
            padding: 25px;
            border-radius: 10px;
            width: 480px;
            max-width: 90%;
            box-shadow: 0 5px 20px rgba(0,0,0,0.4);
        }}

        .modal-content h3 {{ margin-top: 0; color: #1f4e78; border-bottom: 2px solid #2980b9; padding-bottom: 8px; }}
        .modal-field {{ margin-bottom: 15px; }}
        .modal-field label {{ display: block; font-weight: bold; font-size: 12px; margin-bottom: 5px; color: #2c3e50; }}
        .modal-field select {{ width: 100%; padding: 8px; border-radius: 4px; border: 1px solid #ccc; font-size: 13px; }}
        
        .download-action-btn {{
            width: 100%;
            padding: 10px;
            background: #27ae60;
            color: white;
            border: none;
            border-radius: 5px;
            font-weight: bold;
            cursor: pointer;
            font-size: 13px;
        }}
        .download-action-btn:hover {{ background: #219150; }}
        .close-modal-btn {{
            width: 100%;
            padding: 8px;
            background: #7f8c8d;
            color: white;
            border: none;
            border-radius: 5px;
            margin-top: 8px;
            cursor: pointer;
            font-size: 12px;
        }}
    </style>
</head>
<body>

<div id="map"></div>

<div class="top-bar">
    <button class="report-btn" onclick="openReportModal()">📊 Reports & CSV Downloads</button>
</div>

<div class="filter-panel">
    <h4>Filter & Search Farms</h4>
    <div class="filter-group">
        <label for="farmerSearch">🔍 Search Farmer Name</label>
        <input type="text" id="farmerSearch" placeholder="Type farmer name..." oninput="applyFilters()" />
    </div>
    <div class="filter-group">
        <label for="countySelect">County Group</label>
        <select id="countySelect" onchange="onCountyChange()">
            <option value="ALL">All Counties</option>
        </select>
    </div>
    <div class="filter-group">
        <label for="dateSelect">Submission Date</label>
        <select id="dateSelect" onchange="applyFilters()">
            <option value="ALL">All Dates</option>
        </select>
    </div>
    <div class="filter-group">
        <label for="officerSelect">Field Officer</label>
        <select id="officerSelect" onchange="applyFilters()">
            <option value="ALL">All Field Officers</option>
        </select>
    </div>
    <button class="reset-btn" onclick="resetFilters()">Reset Filters</button>
</div>

<div class="summary-panel" id="summaryContainer">
    <h4>Cluster Total Acreage Breakdown</h4>
    <div id="summaryTableDiv">Select filters to view stats</div>
</div>

<div class="modal-overlay" id="reportModal">
    <div class="modal-content">
        <h3>📊 Verification & Audit Reports</h3>
        
        <div class="modal-field">
            <label for="reportTypeSelect">Select Audit Report List</label>
            <select id="reportTypeSelect">
                <option value="coverage">1. Polygon Plough Coverage Rating (%)</option>
                <option value="unvisited">2. Ploughed Farmers with NO Tractor Tracks</option>
                <option value="duplicate">3. Duplicate Shared Polygon Coordinates</option>
                <option value="overlap">4. Overlapping Lands (>5% Area Overlap)</option>
                <option value="unmapped">5. Ploughed Track Areas Without Mapped Farmers</option>
            </select>
        </div>

        <div class="modal-field">
            <label for="reportDateSelect">Select Date Filter</label>
            <select id="reportDateSelect">
                <option value="ALL">All Dates (Full Dataset List)</option>
            </select>
        </div>

        <button class="download-action-btn" onclick="exportSelectedReport()">📥 Download CSV Report</button>
        <button class="close-modal-btn" onclick="closeReportModal()">Close</button>
    </div>
</div>

<script>
    var map = L.map('map').setView([-3.3, 38.5], 10);

    map.createPane('polygonsPane');
    map.getPane('polygonsPane').style.zIndex = 400;

    map.createPane('linesPane');
    map.getPane('linesPane').style.zIndex = 500;

    var googleHybrid = L.tileLayer('https://mt1.google.com/vt/lyrs=y&x={{x}}&y={{y}}&z={{z}}', {{
        maxZoom: 20,
        attribution: '© Google Maps'
    }}).addTo(map);

    var cartoPositron = L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
        maxZoom: 19,
        attribution: '© OpenStreetMap © CARTO'
    }});

    var rawGeoData = {geojson_str};
    var kmlData = {kml_geojson_str};
    var auditReportsData = {reports_json_str};
    
    var geojsonLayer;
    var tractorLayerGroup = L.layerGroup().addTo(map);

    if (kmlData && kmlData.features && kmlData.features.length > 0) {{
        L.geoJSON(kmlData, {{
            pane: 'linesPane',
            interactive: false,
            style: function(feature) {{
                var color = (feature.properties && feature.properties.Track_Color) ? feature.properties.Track_Color : '#e74c3c';
                return {{
                    color: color,
                    weight: 3.5,
                    opacity: 0.85,
                    className: 'leaflet-tractor-line'
                }};
            }}
        }}).addTo(tractorLayerGroup);
    }}

    var baseMaps = {{ "Google Hybrid": googleHybrid, "Light Map (Carto)": cartoPositron }};
    var overlayMaps = {{ "Tractor KML Tracks": tractorLayerGroup }};
    L.control.layers(baseMaps, overlayMaps).addTo(map);

    function defaultStyle() {{
        return {{
            pane: 'polygonsPane',
            fillColor: "#2ecc71",
            weight: 2,
            opacity: 1,
            color: "#27ae60",
            fillOpacity: 0.6
        }};
    }}

    function highlightStyle() {{
        return {{
            fillColor: "#f1c40f",
            weight: 3,
            color: "#f39c12",
            fillOpacity: 0.8
        }};
    }}

    function onEachFeature(feature, layer) {{
        var p = feature.properties;
        var popupHtml = `
            <div class="popup-card">
                <h3>${{p['Farmer Name'] || 'Unknown Farmer'}}</h3>
                <span class="field-label">Location:</span> ${{p.County_Group || p.County || ''}}, ${{p.Subcounty || ''}}, ${{p.Ward || ''}}, ${{p.Village || ''}}<br>
                
                <div class="officer-note">
                    <strong>👨‍🌾 Field Officer Summary:</strong><br>
                    <strong>Officer:</strong> ${{p['Field Officer'] || 'N/A'}}<br>
                    <strong>Date:</strong> ${{p.Clean_Date}}<br>
                    <strong>Total Daily Officer Acres:</strong> ${{p.Total_Acres}} Acres (${{p.Unique_Farmers}} Farmers)<br>
                    <hr style="border:0; border-top:1px dashed #bdc3c7; margin:4px 0;">
                    <strong>Tractor Breakdown:</strong><br>
                    ${{p.Tractor_Breakdown || 'N/A'}}
                </div>
                
                <span class="field-label">Tractor Owner:</span> ${{p['Tractor Owner'] || 'N/A'}}<br>
                <span class="field-label">Tractor Driver:</span> ${{p['Tractor Driver'] || 'N/A'}} (${{p['Tractor Driver - Number Plate'] || 'No Plate'}})<br>
                <span class="field-label">Acres Ploughed (This Farm):</span> ${{p.Acres || '0'}} Acres<br>
                <span class="field-label">Seed Variety:</span> ${{p['Seed Variety'] || 'N/A'}}<br>
                <span class="field-label">Soil / Farm Type:</span> ${{p['Soil Type'] || 'N/A'}} / ${{p['Farm Type'] || 'N/A'}}<br>
                <span class="field-label">Comments:</span> ${{p.Comments || 'None'}}
            </div>
        `;
        layer.bindPopup(popupHtml);
        layer.on({{
            mouseover: function (e) {{ e.target.setStyle(highlightStyle()); }},
            mouseout: function (e) {{ geojsonLayer.resetStyle(e.target); }}
        }});
    }}

    function renderFeatures(data, autoOpenSingle = false) {{
        if (geojsonLayer) {{ map.removeLayer(geojsonLayer); }}
        geojsonLayer = L.geoJSON(data, {{ style: defaultStyle, onEachFeature: onEachFeature }}).addTo(map);
        
        if (data.features && data.features.length > 0) {{
            map.fitBounds(geojsonLayer.getBounds(), {{ padding: [30, 30] }});
            if (autoOpenSingle && data.features.length === 1) {{
                geojsonLayer.eachLayer(function(layer) {{
                    layer.openPopup();
                }});
            }}
        }}
    }}

    function populateDropdowns() {{
        var countySel = document.getElementById('countySelect');
        var counties = new Set();
        rawGeoData.features.forEach(f => {{
            if (f.properties.County_Group) counties.add(f.properties.County_Group);
        }});

        countySel.innerHTML = '<option value="ALL">All Counties</option>';
        Array.from(counties).sort().forEach(c => {{
            countySel.innerHTML += `<option value="${{c}}">${{c}}</option>`;
        }});

        updateCascadingFilters();
    }}

    function onCountyChange() {{
        updateCascadingFilters();
        applyFilters();
    }}

    function updateCascadingFilters() {{
        var selectedCounty = document.getElementById('countySelect').value;
        var dateSel = document.getElementById('dateSelect');
        var officerSel = document.getElementById('officerSelect');
        var reportDateSel = document.getElementById('reportDateSelect');

        var currentSelectedDate = dateSel.value;
        var currentSelectedOfficer = officerSel.value;

        var availableDates = new Set();
        var availableOfficers = new Set();

        rawGeoData.features.forEach(f => {{
            var p = f.properties;
            if (selectedCounty === "ALL" || p.County_Group === selectedCounty) {{
                if (p.Clean_Date) availableDates.add(p.Clean_Date);
                if (p['Field Officer']) availableOfficers.add(p['Field Officer']);
            }}
        }});

        dateSel.innerHTML = '<option value="ALL">All Dates</option>';
        reportDateSel.innerHTML = '<option value="ALL">All Dates (Full Dataset List)</option>';

        Array.from(availableDates).sort().forEach(d => {{
            var sel = (d === currentSelectedDate) ? 'selected' : '';
            dateSel.innerHTML += `<option value="${{d}}" ${{sel}}>${{d}}</option>`;
            reportDateSel.innerHTML += `<option value="${{d}}">${{d}}</option>`;
        }});

        officerSel.innerHTML = '<option value="ALL">All Field Officers</option>';
        Array.from(availableOfficers).sort().forEach(o => {{
            var sel = (o === currentSelectedOfficer) ? 'selected' : '';
            officerSel.innerHTML += `<option value="${{o}}" ${{sel}}>${{o}}</option>`;
        }});
    }}

    function updateSummaryTable(filteredFeatures) {{
        var groups = {{}};
        var grandFarms = 0;
        var grandAcres = 0;

        filteredFeatures.forEach(f => {{
            var p = f.properties;
            var cg = p.County_Group || "Other";
            var ward = p.Ward || p.Subcounty || "General";
            var acres = Number(p.Acres) || 0;

            if (!groups[cg]) groups[cg] = {{}};
            if (!groups[cg][ward]) groups[cg][ward] = {{ farms: 0, acres: 0 }};

            groups[cg][ward].farms += 1;
            groups[cg][ward].acres += acres;
        }});

        var html = '<table class="summary-table"><thead><tr><th>County Group</th><th>Cluster / Ward</th><th>Farms Mapped</th><th>Reported Acres</th></tr></thead><tbody>';

        Object.keys(groups).sort().forEach(cg => {{
            var wards = groups[cg];
            var countyTotalFarms = 0;
            var countyTotalAcres = 0;
            var firstRow = true;

            Object.keys(wards).sort().forEach(w => {{
                var count = wards[w].farms;
                var ac = wards[w].acres;
                countyTotalFarms += count;
                countyTotalAcres += ac;

                html += `<tr>
                    <td>${{firstRow ? `<strong>${{cg}}</strong>` : ''}}</td>
                    <td>${{w}}</td>
                    <td>${{count}}</td>
                    <td>${{ac.toFixed(2)}}</td>
                </tr>`;
                firstRow = false;
            }});

            html += `<tr class="county-subtotal">
                <td colspan="2">TOTAL ${{cg.toUpperCase()}}</td>
                <td>${{countyTotalFarms}}</td>
                <td>${{countyTotalAcres.toFixed(2)}}</td>
            </tr>`;

            grandFarms += countyTotalFarms;
            grandAcres += countyTotalAcres;
        }});

        html += `<tr class="grand-total">
            <td colspan="2">GRAND TOTAL</td>
            <td>${{grandFarms}}</td>
            <td>${{grandAcres.toFixed(2)}}</td>
        </tr></tbody></table>`;

        document.getElementById('summaryTableDiv').innerHTML = html;
    }}

    function applyFilters() {{
        var searchQuery = document.getElementById('farmerSearch').value.trim().toLowerCase();
        var selectedCounty = document.getElementById('countySelect').value;
        var selectedDate = document.getElementById('dateSelect').value;
        var selectedOfficer = document.getElementById('officerSelect').value;

        var filtered = rawGeoData.features.filter(f => {{
            var p = f.properties;
            var farmerName = (p['Farmer Name'] || '').toLowerCase();
            
            var matchSearch = (searchQuery === "") || farmerName.includes(searchQuery);
            var matchCounty = (selectedCounty === "ALL") || (p.County_Group === selectedCounty);
            var matchDate = (selectedDate === "ALL") || (p.Clean_Date === selectedDate);
            var matchOfficer = (selectedOfficer === "ALL") || (p['Field Officer'] === selectedOfficer);
            
            return matchSearch && matchCounty && matchDate && matchOfficer;
        }});

        renderFeatures({{ type: "FeatureCollection", features: filtered }}, searchQuery !== "");
        updateSummaryTable(filtered);
    }}

    function resetFilters() {{
        document.getElementById('farmerSearch').value = "";
        document.getElementById('countySelect').value = "ALL";
        updateCascadingFilters();
        renderFeatures(rawGeoData);
        updateSummaryTable(rawGeoData.features);
    }}

    function openReportModal() {{
        document.getElementById('reportModal').style.display = 'flex';
    }}

    function closeReportModal() {{
        document.getElementById('reportModal').style.display = 'none';
    }}

    function exportSelectedReport() {{
        var rType = document.getElementById('reportTypeSelect').value;
        var rDate = document.getElementById('reportDateSelect').value;

        var keyMap = {{
            "coverage": "coverage_report",
            "unvisited": "unvisited_report",
            "duplicate": "duplicate_report",
            "overlap": "overlap_report",
            "unmapped": "unmapped_report"
        }};

        var dataKey = keyMap[rType];
        var dataset = auditReportsData[dataKey] || [];

        if (rDate !== "ALL") {{
            dataset = dataset.filter(row => row.Clean_Date === rDate);
        }}

        if (dataset.length === 0) {{
            alert("No records found for the selected report and date filter.");
            return;
        }}

        var headers = Object.keys(dataset[0]);
        var csvRows = [headers.join(",")];

        dataset.forEach(row => {{
            var values = headers.map(h => {{
                var val = row[h] === null || row[h] === undefined ? "" : String(row[h]);
                return `"${{val.replace(/"/g, '""')}}"`;
            }});
            csvRows.push(values.join(","));
        }});

        var csvString = csvRows.join("\\n");
        var blob = new Blob([csvString], {{ type: 'text/csv;charset=utf-8;' }});
        var link = document.createElement("a");
        var url = URL.createObjectURL(blob);
        link.setAttribute("href", url);
        link.setAttribute("download", `${{rType}}_audit_report_${{rDate}}.csv`);
        link.style.visibility = 'hidden';
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    }}

    populateDropdowns();
    renderFeatures(rawGeoData);
    updateSummaryTable(rawGeoData.features);
</script>
</body>
</html>
"""

    with open(output_html_path, "w", encoding="utf-8") as f:
        f.write(html_content)


# Execute full processing script
generate_farm_map(
    "farms_polygons_sep_week1_all.gpkg",
    "tractor_kmls_week1",
    "farm_dashboard_sep_week1_all_5%.html",
)

Processing 28 KML track files...
Successfully processed 27 track elements across 26 unique tractors/files.
